# GuppyLM Workshop Notebook

**Estimated time: ~5 minutes on free Colab T4**

This notebook is production-ready for a live workshop and runs top-to-bottom.


## 1. Setup & Introduction

Train a tiny LLM with a custom personality in 5 minutes.


In [1]:
# Install core dependencies for training + chat UI
!pip -q install torch transformers datasets "gradio>=4.44" sentencepiece ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 36.1 MB/s eta 0:00:00


In [2]:
import os
import gc
import json
import time
import math
import random
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import sentencepiece as spm
import ipywidgets as widgets
from IPython.display import display
import gradio as gr

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

WORK_DIR = "/content/guppylm_workshop"
os.makedirs(WORK_DIR, exist_ok=True)
print(f"Working directory: {WORK_DIR}")
print("Setup complete.")


Working directory: /content/guppylm_workshop
Setup complete.


## 2. Model Architecture

Exact GuppyLM vanilla transformer architecture from the original repo (Attention/FFN/Block/GuppyLM implementation preserved for compatibility), with workshop config values:
- `vocab_size=32000`
- `d_model=256`
- `n_heads=4`
- `n_layers=4`
- `max_seq_len=256`


In [3]:
@dataclass
class GuppyConfig:
    vocab_size: int = 32000
    max_seq_len: int = 256
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    ffn_hidden: int = 512
    dropout: float = 0.1

    # Special tokens
    pad_id: int = 0
    bos_id: int = 1
    eos_id: int = 2


class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.out = nn.Linear(config.d_model, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float("-inf"))
        attn = self.dropout(F.softmax(attn, dim=-1))
        return self.out((attn @ v).transpose(1, 2).contiguous().view(B, T, C))


class FFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.up = nn.Linear(config.d_model, config.ffn_hidden)
        self.down = nn.Linear(config.ffn_hidden, config.d_model)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.down(F.relu(self.up(x))))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.d_model)
        self.attn = Attention(config)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.ffn = FFN(config)

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x


class GuppyLM(nn.Module):
    def __init__(self, config: GuppyConfig):
        super().__init__()
        self.config = config

        self.tok_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.pos_emb = nn.Embedding(config.max_seq_len, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layers)])
        self.norm = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # tie weights

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)

        for block in self.blocks:
            x = block(x, mask)

        logits = self.lm_head(self.norm(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1),
                ignore_index=self.config.pad_id,
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=64, temperature=0.7, top_k=50):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            if next_id.item() == self.config.eos_id:
                break
        return idx

    def param_count(self):
        return sum(p.numel() for p in self.parameters())


cfg = GuppyConfig()
model = GuppyLM(cfg)
print(f"GuppyLM params: {model.param_count():,} ({model.param_count()/1e6:.2f}M)")


GuppyLM params: 10,366,464 (10.37M)


        ## 3. Load & Prepare Dataset

        - Supports JSONL in `instruction` / `input` / `output` format.
        - If `input` is empty, uses `instruction` directly as the prompt.
        - Tokenizer uses SentencePiece BPE.
        - Dataset formats each sample as `Instruction: {instruction}
Response: {output}`.


In [4]:
class SPMTokenizer:
    def __init__(self, model_path: str):
        self.model_path = model_path
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(model_path)

    def encode(self, text: str, add_bos: bool = False, add_eos: bool = True) -> List[int]:
        ids = self.sp.encode(text, out_type=int)
        if add_bos:
            ids = [self.sp.bos_id()] + ids
        if add_eos:
            ids = ids + [self.sp.eos_id()]
        return ids

    def decode(self, ids: List[int]) -> str:
        return self.sp.decode(ids)

    @property
    def pad_id(self) -> int:
        return self.sp.pad_id()

    @property
    def bos_id(self) -> int:
        return self.sp.bos_id()

    @property
    def eos_id(self) -> int:
        return self.sp.eos_id()


def _to_text(value) -> str:
    if value is None:
        return ""
    return str(value).strip()


def normalize_record(rec: Dict) -> Dict[str, str]:
    instruction = _to_text(rec.get("instruction", ""))
    input_text = _to_text(rec.get("input", ""))
    output = _to_text(rec.get("output", ""))
    if not instruction:
        instruction = _to_text(rec.get("prompt", ""))
    if not output:
        output = _to_text(rec.get("response", ""))

    if not instruction or not output:
        raise ValueError("Each JSONL row must contain instruction and output (or prompt/response).")

    if input_text:
        final_instruction = f"{instruction}\nInput: {input_text}"
    else:
        final_instruction = instruction

    return {
        "instruction": final_instruction,
        "output": output,
    }


def format_instruction_response(rec: Dict[str, str]) -> str:
    return f"Instruction: {rec['instruction']}\nResponse: {rec['output']}"


def parse_jsonl_bytes(content: bytes, source_name: str = "uploaded.jsonl") -> List[Dict[str, str]]:
    rows = []
    for i, line in enumerate(content.decode("utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        try:
            raw = json.loads(line)
            rows.append(normalize_record(raw))
        except Exception as e:
            raise ValueError(f"{source_name}: bad JSONL at line {i}: {e}")
    if not rows:
        raise ValueError(f"{source_name}: file is empty or invalid.")
    return rows


def train_sentencepiece_tokenizer(texts: List[str], save_dir: str, vocab_size: int = 32000) -> str:
    os.makedirs(save_dir, exist_ok=True)
    corpus_path = os.path.join(save_dir, "spm_corpus.txt")
    model_prefix = os.path.join(save_dir, "spm")

    with open(corpus_path, "w", encoding="utf-8") as f:
        for t in texts:
            f.write(t.replace("\n", " ") + "\n")

    spm.SentencePieceTrainer.train(
        input=corpus_path,
        model_prefix=model_prefix,
        model_type="bpe",
        vocab_size=vocab_size,
        hard_vocab_limit=False,
        pad_id=0,
        bos_id=1,
        eos_id=2,
        unk_id=3,
        pad_piece="<pad>",
        bos_piece="<|im_start|>",
        eos_piece="<|im_end|>",
    )
    return model_prefix + ".model"


class InstructionDataset(Dataset):
    def __init__(self, records: List[Dict[str, str]], tokenizer: SPMTokenizer, max_len: int = 256):
        self.samples = []
        for rec in records:
            text = format_instruction_response(rec)
            ids = tokenizer.encode(text, add_bos=True, add_eos=True)
            if len(ids) < 2:
                continue
            ids = ids[:max_len]
            self.samples.append(ids)

        if not self.samples:
            raise ValueError("No usable samples after tokenization.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        ids = self.samples[idx]
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y


def collate_batch(batch, pad_id: int = 0):
    xs, ys = zip(*batch)
    max_len = max(x.shape[0] for x in xs)
    bx = torch.full((len(xs), max_len), pad_id, dtype=torch.long)
    by = torch.full((len(xs), max_len), pad_id, dtype=torch.long)
    for i, (x, y) in enumerate(zip(xs, ys)):
        bx[i, : x.shape[0]] = x
        by[i, : y.shape[0]] = y
    return bx, by


print("Dataset + tokenizer utilities ready.")


Dataset + tokenizer utilities ready.


## 4. Training Loop

Simple training loop with AdamW. Uses Colab T4 GPU if available, otherwise CPU fallback. Prints loss every 50 steps. Defaults are configurable:
- `lr=3e-4`
- `batch_size=8`
- `epochs=3`


In [5]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        print(f"Using GPU: {name}")
        return torch.device("cuda")
    print("Warning: GPU not detected. Falling back to CPU (training will be slower).")
    return torch.device("cpu")


def train_model(
    records: List[Dict[str, str]],
    run_name: str,
    lr: float = 3e-4,
    batch_size: int = 8,
    epochs: int = 3,
    config: Optional[GuppyConfig] = None,
):
    if config is None:
        config = GuppyConfig()

    device = get_device()
    run_dir = os.path.join(WORK_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    texts = [format_instruction_response(r) for r in records]
    tokenizer_model_path = train_sentencepiece_tokenizer(
        texts=texts,
        save_dir=run_dir,
        vocab_size=config.vocab_size,
    )
    tokenizer = SPMTokenizer(tokenizer_model_path)

    dataset = InstructionDataset(records, tokenizer, max_len=config.max_seq_len)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=lambda b: collate_batch(b, pad_id=tokenizer.pad_id),
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    model = GuppyLM(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    print(f"Training samples: {len(dataset)}")
    print(f"Steps per epoch: {len(loader)}")

    step = 0
    model.train()
    for epoch in range(epochs):
        running = 0.0
        running_count = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            _, loss = model(x, y)
            loss.backward()
            optimizer.step()

            running += loss.item()
            running_count += 1
            if step % 50 == 0:
                avg = running / max(1, running_count)
                print(f"epoch={epoch+1}/{epochs} step={step} loss={loss.item():.4f} avg_recent={avg:.4f}")
                running = 0.0
                running_count = 0
            step += 1

    ckpt_path = os.path.join(run_dir, "final_model.pt")
    cfg_path = os.path.join(run_dir, "config.json")
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": asdict(config),
            "tokenizer_model_path": tokenizer_model_path,
        },
        ckpt_path,
    )
    with open(cfg_path, "w", encoding="utf-8") as f:
        json.dump(asdict(config), f, indent=2)

    print(f"Checkpoint saved: {ckpt_path}")
    return {
        "run_dir": run_dir,
        "checkpoint_path": ckpt_path,
        "tokenizer_model_path": tokenizer_model_path,
        "config": asdict(config),
    }


## 5. Chat Interface (Gradio)

Loads the trained model and serves chat with `temperature` and `max_tokens` controls.

Uses `gr.Blocks` with an explicit **Send** button (more reliable in Colab iframes than `gr.ChatInterface`).
Launched with `share=True` to produce a public URL for workshop participants on locked laptops.


In [ ]:
LATEST_TRAINING = None
CHAT_IFACE = None

def _load_checkpoint_for_chat(training_info: Dict):
    ckpt = torch.load(training_info["checkpoint_path"], map_location="cpu")
    cfg = GuppyConfig(**ckpt["config"])
    model = GuppyLM(cfg)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    model.eval()
    tokenizer = SPMTokenizer(training_info["tokenizer_model_path"])
    return model, tokenizer, cfg

def build_chat_fn(training_info: Dict):
    model, tokenizer, cfg = _load_checkpoint_for_chat(training_info)
    device = get_device()
    model = model.to(device)

    @torch.no_grad()
    def chat(message: str, history: List[Dict[str, str]], temperature: float, max_tokens: int):
        turns = []
        for turn in history[-6:]:
            role = "User" if turn['role'] == 'user' else "Assistant"
            turns.append(f"{role}: {turn['content']}")
        turns.append(f"User: {message}\nAssistant:")
        prompt = "\n".join(turns)

        ids = tokenizer.encode(prompt, add_bos=True, add_eos=False)
        x = torch.tensor([ids[-cfg.max_seq_len:]], dtype=torch.long, device=device)

        out = model.generate(
            x,
            max_new_tokens=int(max_tokens),
            temperature=float(temperature),
            top_k=50,
        )
        gen_ids = out[0].tolist()[x.shape[1]:]
        if cfg.eos_id in gen_ids:
            gen_ids = gen_ids[:gen_ids.index(cfg.eos_id)]

        text = tokenizer.decode(gen_ids).strip() or "(no output)"
        return text

    return chat

def _build_blocks_interface(fn):
    """Build a gr.Blocks chat UI with an explicit Send button.

    gr.ChatInterface can lose the submit button when rendered inside
    Colab's iframe. Using gr.Blocks with explicit components avoids this.
    """
    with gr.Blocks(title="GuppyLM Workshop") as demo:
        gr.Markdown("## GuppyLM Workshop Chat")
        chatbot = gr.Chatbot(type="messages", height=400)
        with gr.Row():
            msg_box = gr.Textbox(
                placeholder="Type your message\u2026",
                show_label=False,
                scale=4,
                autofocus=True,
            )
            send_btn = gr.Button("Send", variant="primary", scale=1)
        with gr.Row():
            temperature = gr.Slider(0.1, 1.5, value=0.8, label="Temperature")
            max_tokens = gr.Slider(16, 256, value=96, step=8, label="Max Tokens")
        clear_btn = gr.Button("Clear Chat")

        def respond(user_msg, chat_history, temp, mtokens):
            if not user_msg or not user_msg.strip():
                return "", chat_history
            reply = fn(user_msg, chat_history, temp, mtokens)
            chat_history.append({"role": "user", "content": user_msg})
            chat_history.append({"role": "assistant", "content": reply})
            return "", chat_history

        def clear_chat():
            return [], ""

        # Wire submit via button click AND Enter key (textbox submit)
        send_btn.click(
            fn=respond,
            inputs=[msg_box, chatbot, temperature, max_tokens],
            outputs=[msg_box, chatbot],
        )
        msg_box.submit(
            fn=respond,
            inputs=[msg_box, chatbot, temperature, max_tokens],
            outputs=[msg_box, chatbot],
        )
        clear_btn.click(fn=clear_chat, inputs=[], outputs=[chatbot, msg_box])

    return demo

def launch_chat_interface(training_info: Optional[Dict] = None):
    global CHAT_IFACE
    if training_info is None:
        training_info = LATEST_TRAINING
    if training_info is None:
        raise ValueError("No trained model found yet.")

    fn = build_chat_fn(training_info)

    # Build a Blocks-based interface with an explicit Send button
    # (more robust than ChatInterface inside Colab iframes)
    CHAT_IFACE = _build_blocks_interface(fn)

    launch_result = CHAT_IFACE.launch(share=True, inline=True, debug=False)

    share_url = getattr(CHAT_IFACE, 'share_url', None)
    print("\n" + "=" * 70)
    print("Public URL:", share_url or "Check output above for URL.")
    print("Type a message and click Send (or press Enter).")
    print("=" * 70)
    return CHAT_IFACE, share_url

print("Chat interface builder updated.")


## 6. Quick Demo with Pre-loaded Personas

Persona dropdown options:
1. Tech Grandma (loads from `tech_grandma.jsonl` if uploaded)
2. Sarcastic Coach (20 inline fallback examples)
3. Noir Detective (20 inline fallback examples)


In [7]:
def build_sarcastic_coach_examples() -> List[Dict[str, str]]:
    prompts = [
        "I skipped leg day again.",
        "How do I stay consistent?",
        "I am too tired to train.",
        "Should I warm up?",
        "I hate cardio.",
        "How much protein should I eat?",
        "I have no motivation today.",
        "Can I build muscle in 20 minutes?",
        "What is progressive overload?",
        "I only trained once this week.",
        "Is rest day laziness?",
        "How do I fix my form?",
        "I keep quitting after two weeks.",
        "Can I out-train bad sleep?",
        "What do I do when I plateau?",
        "Do I need supplements?",
        "How many reps should I do?",
        "I am embarrassed at the gym.",
        "Can I get fit at home?",
        "Give me a pep talk.",
    ]
    outputs = [
        "Amazing strategy. Avoid legs and hope stairs become optional.",
        "Make the plan so small your excuses look dramatic.",
        "Train anyway, just reduce intensity and keep the streak alive.",
        "Yes. Five minutes now saves two weeks of fake injuries.",
        "Then walk fast and complain less. Efficient and traditional.",
        "Enough to recover. Start near 1.6 to 2.2 g/kg bodyweight.",
        "Motivation is cute. Habits are what show up on schedule.",
        "You can build momentum in 20 minutes. Do compound lifts first.",
        "Add a little weight, reps, or control over time.",
        "Great. Beat that number next week and call it progress.",
        "Rest day is training for tomorrow's quality work.",
        "Lower the load, film one set, and fix one cue at a time.",
        "Shrink the goal. Two weeks becomes day one repeated.",
        "No. Sleep is your legal performance enhancer.",
        "Change one variable only and track it for two weeks.",
        "Supplements are optional. Food, sleep, and consistency are not.",
        "Pick a rep range and own it with good form.",
        "Everyone started somewhere. Confidence is reps in public.",
        "Yes. Squat, hinge, push, pull, carry. Repeat weekly.",
        "You are not behind. You are one session away from momentum.",
    ]
    return [{"instruction": p, "input": "", "output": o} for p, o in zip(prompts, outputs)]


def build_noir_detective_examples() -> List[Dict[str, str]]:
    prompts = [
        "What happened downtown tonight?",
        "Who can I trust in this city?",
        "I heard a gunshot near the pier.",
        "Why does everyone lie to me?",
        "Any leads on the missing briefcase?",
        "Describe the rain tonight.",
        "What does the bartender know?",
        "Should I confront the mayor?",
        "How do I read a suspect?",
        "I found a matchbook clue.",
        "What is your hunch?",
        "Do you ever sleep?",
        "Why keep chasing cold cases?",
        "What did the alley tell you?",
        "How bad is corruption here?",
        "What is in your notebook?",
        "Give me the short version.",
        "Any advice before I knock?",
        "Who is the villain?",
        "How does this story end?",
    ]
    outputs = [
        "Downtown coughed up secrets and blood, in that order.",
        "Trust is expensive here. Buy in cash, never in promises.",
        "Then the pier just wrote us an invitation in gunpowder.",
        "Because truth has no lobbyists and lies have office hours.",
        "Three leads. Two are dead ends. One is scared of daylight.",
        "Rain hit like static, washing prints and feeding rumors.",
        "Enough to retire early, not enough to stay alive.",
        "Not before dawn. Night makes brave men sloppy.",
        "Watch the eyes first, then the hands, then the silence.",
        "Good. Small clues wear loud shoes in quiet rooms.",
        "The clean tie with dirty cuffs did it.",
        "Only when the city forgets my name for an hour.",
        "Cold cases are debts the city keeps dodging.",
        "It said hurry. It always says hurry.",
        "Bad enough to need accountants, not cops.",
        "Names, times, and the lies between them.",
        "Someone stole something small and panicked big.",
        "Count exits before questions. Always.",
        "The one smiling while everyone else sweats.",
        "Messy. True. Expensive.",
    ]
    return [{"instruction": p, "input": "", "output": o} for p, o in zip(prompts, outputs)]


PERSONA_OPTIONS = ["Tech Grandma", "Sarcastic Coach", "Noir Detective"]


def get_persona_records(persona_name: str, uploaded_files: Dict[str, List[Dict[str, str]]]) -> List[Dict[str, str]]:
    if persona_name == "Sarcastic Coach":
        return [normalize_record(r) for r in build_sarcastic_coach_examples()]
    if persona_name == "Noir Detective":
        return [normalize_record(r) for r in build_noir_detective_examples()]
    if "tech_grandma.jsonl" in uploaded_files:
        return uploaded_files["tech_grandma.jsonl"]
    raise ValueError("Tech Grandma selected, but tech_grandma.jsonl is not uploaded yet.")


print("Persona helpers ready.")


Persona helpers ready.


## 7. Custom Dataset Upload

- Upload custom JSONL
- Preview first 5 entries
- Train button

You can train from a selected persona OR a custom uploaded JSONL.


In [8]:
UPLOADED_JSONL: Dict[str, List[Dict[str, str]]] = {}


def _extract_uploaded_files(file_upload_widget) -> Dict[str, bytes]:
    data = {}
    value = file_upload_widget.value
    if isinstance(value, dict):
        for fname, meta in value.items():
            content = meta.get("content", b"")
            if isinstance(content, memoryview):
                content = content.tobytes()
            data[fname] = content
    elif isinstance(value, tuple):
        for item in value:
            fname = item.get("name")
            content = item.get("content", b"")
            if isinstance(content, memoryview):
                content = content.tobytes()
            data[fname] = content
    return data


def refresh_uploaded_jsonl(file_upload_widget):
    UPLOADED_JSONL.clear()
    raw_files = _extract_uploaded_files(file_upload_widget)
    for fname, content in raw_files.items():
        if not fname.lower().endswith(".jsonl"):
            continue
        UPLOADED_JSONL[fname] = parse_jsonl_bytes(content, source_name=fname)


def preview_records(records: List[Dict[str, str]], n: int = 5) -> str:
    lines = []
    for i, rec in enumerate(records[:n], start=1):
        lines.append(f"{i}. instruction={rec['instruction'][:120]!r}")
        lines.append(f"   output={rec['output'][:120]!r}")
    return "\n".join(lines)


def choose_training_records(persona_name: str, prefer_custom: bool) -> Tuple[str, List[Dict[str, str]]]:
    custom_candidates = [k for k in UPLOADED_JSONL.keys() if k != "tech_grandma.jsonl"]
    if prefer_custom:
        if not custom_candidates:
            raise ValueError("Custom dataset mode is enabled but no custom .jsonl file was uploaded.")
        selected = custom_candidates[0]
        return selected, UPLOADED_JSONL[selected]
    return persona_name, get_persona_records(persona_name, UPLOADED_JSONL)


persona_dropdown = widgets.Dropdown(options=PERSONA_OPTIONS, value="Sarcastic Coach", description="Persona:")
use_custom_checkbox = widgets.Checkbox(value=False, description="Use custom uploaded JSONL")
uploader = widgets.FileUpload(accept=".jsonl", multiple=True, description="Upload JSONL")

lr_widget = widgets.FloatText(value=3e-4, description="LR:")
batch_widget = widgets.IntText(value=8, description="Batch:")
epochs_widget = widgets.IntText(value=3, description="Epochs:")

preview_btn = widgets.Button(description="Preview first 5", button_style="info")
train_btn = widgets.Button(description="Train", button_style="success")
status_out = widgets.Output(layout={"border": "1px solid #ddd", "padding": "8px"})


def on_preview_clicked(_):
    with status_out:
        status_out.clear_output()
        try:
            refresh_uploaded_jsonl(uploader)
            source_name, records = choose_training_records(persona_dropdown.value, use_custom_checkbox.value)
            print(f"Using dataset: {source_name} ({len(records)} records)")
            print(preview_records(records, n=5))
        except Exception as e:
            print(f"Preview error: {e}")


def on_train_clicked(_):
    global LATEST_TRAINING
    with status_out:
        status_out.clear_output()
        try:
            refresh_uploaded_jsonl(uploader)
            source_name, records = choose_training_records(persona_dropdown.value, use_custom_checkbox.value)
            print(f"Training from: {source_name}")
            print(f"Records: {len(records)}")

            run_name = f"run_{int(time.time())}"
            LATEST_TRAINING = train_model(
                records=records,
                run_name=run_name,
                lr=float(lr_widget.value),
                batch_size=int(batch_widget.value),
                epochs=int(epochs_widget.value),
                config=GuppyConfig(),
            )

            print("\nTraining complete.")
            print(json.dumps(LATEST_TRAINING, indent=2))
            print("\nLaunching Gradio with share=True ...")
            _, share_url = launch_chat_interface(LATEST_TRAINING)
            print(f"\nPublic URL: {share_url}")

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception as e:
            print(f"Training error: {e}")


preview_btn.on_click(on_preview_clicked)
train_btn.on_click(on_train_clicked)

ui = widgets.VBox([
    widgets.HTML("<h4>Workshop Trainer</h4>"),
    persona_dropdown,
    use_custom_checkbox,
    uploader,
    widgets.HBox([lr_widget, batch_widget, epochs_widget]),
    widgets.HBox([preview_btn, train_btn]),
    status_out,
])

display(ui)
print("Upload files, preview first 5 entries, then train.")


Upload files, preview first 5 entries, then train.
